# Laboratorium 5: Przetwarzanie tekstu, klasyfikacja tekstu, modelowanie tematów

In [ ]:
%pip install -q sentence-transformers

In [ ]:
import re
import time
from collections import Counter

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from sklearn.datasets import fetch_20newsgroups
from sklearn.decomposition import NMF, LatentDirichletAllocation
from sklearn.dummy import DummyClassifier
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer, ENGLISH_STOP_WORDS
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (ConfusionMatrixDisplay, adjusted_rand_score, classification_report,
                             f1_score, normalized_mutual_info_score)
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import GridSearchCV, StratifiedKFold, cross_validate, train_test_split
from sklearn.naive_bayes import ComplementNB
from sklearn.pipeline import Pipeline
from sklearn.svm import LinearSVC

RANDOM_STATE = 42
BASE = "https://huggingface.co/datasets/allegro/klej-polemo2-in/resolve/main/"
TOKEN = r"(?u)\b\w+\b"  # słowa złożone z liter i cyfr, także jednoliterowe
results = {}  # tu zbieraj wyniki do podsumowania w zadaniu 8

## Zadanie 1: Wczytanie i eksploracja korpusu

### 1.1 Wczytanie danych

In [ ]:
# Wczytaj train.csv, valid.csv i test.csv z adresu BASE (kolumny: sentence, target)
# Wypisz rozmiary zbiorów i unikalne wartości kolumny target

### 1.2 Rozkład klas

In [ ]:
# Zamień wartości target na krótkie etykiety (neg, neu, pos, amb) w kolumnie label
# Policz klasy w każdym zbiorze (liczebności i udziały) oraz duplikaty tekstów

### 1.3 Długości tekstów

In [ ]:
# Dodaj kolumnę n_words (liczba słów) do każdego zbioru
# Opisz rozkład długości (describe), narysuj histogram i wykres pudełkowy wg klasy
# Obejrzyj najkrótsze dokumenty

### 1.4 Przykładowe dokumenty

In [ ]:
# Wypisz po jednym losowym dokumencie z każdej klasy (pierwsze ok. 350 znaków)

### 1.5 Statystyki słownikowe i prawo Zipfa

In [ ]:
# Podziel teksty na słowa wyrażeniem regularnym (małe litery), policz częstości (Counter)
# Podaj liczbę tokenów, rozmiar słownika, udział słów występujących raz, 20 najczęstszych słów
# Narysuj wykres ranga-częstość w skali log-log

### 1.6 Zbiory do klasyfikacji

In [ ]:
# Usuń klasę amb i przygotuj X_train, y_train, X_valid, y_valid, X_test, y_test
# (zbiór testowy nie jest używany do żadnych decyzji aż do końca zadania 3)

## Zadanie 2: Reprezentacja tekstu

### 2.1 Normalizacja i tokenizacja

In [ ]:
# Napisz normalize(text) (małe litery, białe znaki) i tokenize(text) (re.findall)
# Sprawdź na jednym dokumencie

### 2.2 Bag of words

In [ ]:
# CountVectorizer(token_pattern=TOKEN) dopasowany na X_train
# Podaj kształt, liczbę niezerowych elementów, gęstość i pamięć (macierz rzadka vs gęsta)
# Wypisz najczęstsze słowa pierwszego dokumentu wraz z licznikami

### 2.3 TF-IDF

In [ ]:
# TfidfVectorizer(token_pattern=TOKEN) dopasowany na X_train
# Wypisz słowa o najniższym i najwyższym idf oraz najwyżej ważone słowa jednego dokumentu

### 2.4 Rozmiar słownika

In [ ]:
# Zbuduj słownik na X_train dla kilku ustawień (min_df, max_df, ngram_range, analyzer="char_wb")
# Zestaw w tabeli: liczba cech, średnia liczba niezerowych na dokument, czas dopasowania

### 2.5 Fleksja i n-gramy znakowe

In [ ]:
# Sprawdź, czy NLTK SnowballStemmer obsługuje język polski
# Dla odmian słowa "lekarz" porównaj zbiory n-gramów znakowych (char_wb, 3-5), np. miarą Jaccarda

## Zadanie 3: Klasyfikacja sentymentu

### 3.1 Porównanie klasyfikatorów

In [ ]:
# Podziel walidację krzyżową: StratifiedKFold(5, shuffle=True, random_state=RANDOM_STATE)
# Dla wspólnej wektoryzacji TF-IDF (słowa 1-2, min_df=2) porównaj w CV (F1 makro, dokładność):
# DummyClassifier, ComplementNB, LogisticRegression, LinearSVC; każdy jako Pipeline(TfidfVectorizer, klasyfikator)

### 3.2 Porównanie wektoryzacji

In [ ]:
# Dla LinearSVC porównaj w CV warianty wektoryzacji: min_df/max_df, n-gramy słów, sublinear_tf, n-gramy znaków
# Zestaw wyniki w tabeli (F1 makro, odchylenie, czas)

### 3.3 Dobór parametru C

In [ ]:
# GridSearchCV po parametrze clf__C wybranego potoku (LinearSVC), ocena F1 makro w CV na zbiorze treningowym

### 3.4 Ocena na zbiorze walidacyjnym

In [ ]:
# Dopasuj najlepszy potok na całym X_train, oceń na zbiorze walidacyjnym
# classification_report, macierz pomyłek (ConfusionMatrixDisplay), F1 makro zapisz w słowniku results

### 3.5 Zbiór testowy

In [ ]:
# Jednorazowo oceń wybrany model na zbiorze testowym (bez dalszego strojenia)

## Zadanie 4: Interpretacja i błędy

### 4.1 Najważniejsze cechy klas

In [ ]:
# Z najlepszego potoku wyciągnij nazwy cech (TfidfVectorizer) i współczynniki (coef_) klasyfikatora
# Zestaw w tabeli 15 cech o największych współczynnikach dla każdej klasy

### 4.2 Błędne klasyfikacje

In [ ]:
# Zbuduj tabelę błędów na zbiorze walidacyjnym (tekst, etykieta, predykcja, długość)
# Policz odsetek błędów w kwartylach długości (pd.qcut)
# Wypisz kilka błędów, w których model był najbardziej pewny (największy margines decision_function)

## Zadanie 5: Modelowanie tematów

### 5.1 Korpus 20 Newsgroups

In [ ]:
# Pobierz 20 Newsgroups dla 6 kategorii (train, remove=("headers", "footers", "quotes"), shuffle z RANDOM_STATE)
# Usuń dokumenty krótsze niż 15 słów, wypisz liczności kategorii

### 5.2 Wektoryzacja

In [ ]:
# Przygotuj listę stop words (angielska z sklearn + własne słowa ogólne, dodane po obejrzeniu pierwszych tematów)
# NMF pracuje na TF-IDF, LDA na licznikach: zbuduj TfidfVectorizer i CountVectorizer (min_df, max_df, tylko słowa z liter)

### 5.3 NMF i LDA

In [ ]:
# Napisz top_words(components, feature_names, n) zwracającą n najważniejszych słów każdego tematu
# NMF(K=6) na macierzy TF-IDF oraz LatentDirichletAllocation(K=6) na licznikach; wypisz tematy obu metod

### 5.4 Porównanie z kategoriami

In [ ]:
# Dla obu modeli zestaw tabelę krzyżową: dominujący temat dokumentu i prawdziwa kategoria
# Policz ARI i NMI (adjusted_rand_score, normalized_mutual_info_score)

### 5.5 Dobór liczby tematów

In [ ]:
# NMF: dla k z listy policz błąd rekonstrukcji (reconstruction_err_), ARI i NMI względem kategorii
# LDA: podziel dokumenty na treningowe i odłożone, dla kilku k policz perplexity na odłożonych
# Narysuj wykresy zależności od k, a następnie obejrzyj tematy wybranego k i oceń ich interpretowalność

### 5.6 Tematy dla wybranego k

In [ ]:
# Dopasuj NMF dla wybranego k, wypisz tematy i tabelę krzyżową z kategoriami

## Zadanie 6: Podobieństwo dokumentów i wyszukiwarka

### 6.1 Indeks i funkcja wyszukiwania

In [ ]:
# Zbuduj indeks TF-IDF całego train (dwa warianty: słowa i n-gramy znakowe)
# Napisz search(query, vec, index, k) zwracającą k najbliższych dokumentów (cosine_similarity)
# Sprawdź kilka własnych zapytań w obu wariantach

### 6.2 Odmiana wyrazów w zapytaniu

In [ ]:
# Zapytanie z formą fleksyjną rzadko występującą w korpusie: porównaj wyniki wariantu słownego i znakowego
# Znajdź też dokumenty najbardziej podobne do wybranego dokumentu z korpusu

## Zadanie 7: Embeddingi zdań

### 7.1 Podpróba i kodowanie

In [ ]:
# Wylosuj stratyfikowaną podpróbę: 800 dokumentów treningowych i 300 walidacyjnych
# Wczytaj SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2", device="cpu") i zakoduj teksty (zmierz czas)

### 7.2 Porównanie z TF-IDF

In [ ]:
# LogisticRegression na embeddingach vs TF-IDF + LogisticRegression i TF-IDF + LinearSVC na tej samej podpróbie
# Porównaj F1 makro na podpróbie walidacyjnej

## Zadanie 8: Podsumowanie

In [ ]:
# Zbierz w jednej tabeli wyniki z zadań 3 i 7 (słownik results)